# 35. MiT Mix Transformer Encoder 구조

이 노트북은 SegFormer의 encoder인 MiT(Mix Transformer)를 이해하는 단계입니다.

MiT는 ViT처럼 patch token을 사용하지만, segmentation에 맞게 계층적 feature map을 만듭니다. 또한 overlapping patch embedding을 사용해 patch 경계에서 정보가 끊기는 문제를 줄입니다.

이번 노트북의 목표는 다음과 같습니다.

- MiT encoder의 4-stage 구조를 이해합니다.
- overlapping patch embedding의 의미를 확인합니다.
- stage가 깊어질수록 해상도는 줄고 channel은 늘어나는 흐름을 파악합니다.
- SegFormer decoder가 사용할 multi-scale feature의 형태를 정리합니다.

In [ ]:
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.unicode_minus'] = False

## 35-1. MiT의 stage 구조

SegFormer의 MiT encoder는 보통 4개의 stage feature를 출력합니다.

```text
input image
  -> stage 1: H/4  x W/4
  -> stage 2: H/8  x W/8
  -> stage 3: H/16 x W/16
  -> stage 4: H/32 x W/32
```

이 출력들은 decoder에서 같은 해상도로 맞춰진 뒤 결합됩니다.

In [ ]:
input_size = 224
channels = [64, 128, 320, 512]
strides = [4, 8, 16, 32]

for i, (s, c) in enumerate(zip(strides, channels), start=1):
    print(f'stage {i}: {input_size // s} x {input_size // s} x {c}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.axis('off')
for i, (s, c) in enumerate(zip(strides, channels)):
    h = input_size // s
    x = i * 1.9
    size = 1.4 / (i + 1) + 0.25
    ax.add_patch(Rectangle((x, 1 - size / 2), size, size, facecolor='#fef3c7', edgecolor='#d97706', linewidth=2))
    ax.text(x + size / 2, 1, f'{h}x{h}\nC={c}', ha='center', va='center', fontsize=9)
    ax.text(x + size / 2, 0.1, f'Stage {i + 1}', ha='center')
    if i < 3:
        ax.annotate('', xy=(x + 1.65, 1), xytext=(x + size + 0.1, 1), arrowprops=dict(arrowstyle='->'))
ax.set_xlim(-0.2, 7.1)
ax.set_ylim(-0.1, 1.8)
ax.set_title('MiT encoder multi-scale outputs')
plt.show()

## 35-2. Overlapping patch embedding

ViT의 patch embedding은 겹치지 않는 patch를 사용하는 경우가 많습니다. MiT는 convolution처럼 kernel과 stride를 다르게 두어 patch가 서로 겹치게 만듭니다.

예를 들어 kernel size가 7이고 stride가 4이면 인접 token이 입력 영역을 일부 공유합니다.

In [ ]:
def conv_out(size, kernel, stride, padding):
    return (size + 2 * padding - kernel) // stride + 1

for kernel, stride, padding in [(7, 4, 3), (3, 2, 1), (16, 16, 0)]:
    out = conv_out(224, kernel, stride, padding)
    print(f'kernel={kernel}, stride={stride}, padding={padding} -> output {out}x{out}')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.set_xlim(0, 12)
ax.set_ylim(0, 8)
ax.set_aspect('equal')
ax.set_title('overlapping patch embedding 예시')
ax.set_xticks(range(13))
ax.set_yticks(range(9))
ax.grid(alpha=0.3)
patch_specs = [(1, 1, '#93c5fd'), (5, 1, '#86efac'), (1, 5, '#fca5a5')]
for x, y, color in patch_specs:
    ax.add_patch(Rectangle((x, y), 5, 5, facecolor=color, edgecolor='black', alpha=0.45, linewidth=2))
    ax.text(x + 2.5, y + 2.5, 'kernel', ha='center', va='center')
ax.text(1, 7.4, 'stride가 kernel보다 작으면 입력 영역이 겹침', fontsize=10)
plt.show()

## 35-3. MiT block의 구성

MiT block은 큰 틀에서 Transformer block입니다. 다만 attention 계산을 효율화하고, 위치 정보를 별도 positional embedding에 강하게 의존하지 않도록 설계합니다.

```text
feature map
  -> efficient self-attention
  -> Mix-FFN
  -> next feature map
```

SegFormer 논문에서 중요한 포인트는 **positional encoding을 제거해 입력 해상도 변화에 더 유연하게 대응**하려는 설계입니다.

## 정리

- MiT encoder는 segmentation에 필요한 4단계 multi-scale feature를 만듭니다.
- overlapping patch embedding은 patch 경계의 단절을 줄이는 역할을 합니다.
- stage가 깊어질수록 공간 해상도는 줄고 channel 수는 늘어납니다.
- 다음 노트북 `36_Efficient_Self_Attention_이해.ipynb`에서는 MiT의 attention 효율화 방식을 살펴봅니다.